# Bridge — apply the regional model to the 8 dam catchments (Colab)

The application chapter: the trained regional LSTM simulates inflow to the 8
unregulated Guadalquivir dams **as if their catchments were ungauged** (they were
never in training), and we compare monthly volumes (hm³) to the corrected
reconstruction `I_corr` from `bridge_inflow_reconstruction.py`.
This is the exact simulation of the Moroccan use case, validated where truth exists.

**Stages (each cached on Drive → resumable):**
1. Delineate the 8 dam catchments (HydroSHEDS 03DIR, snap to max upstream area at the dam wall)
2. ERA5-Land forcings (yearly batched) + Hargreaves PET
3. Static attributes (same recipe as `pooled_attributes`)
4. Extend `nh_pooled` with `dam_<id>` basins (qobs = NaN)
5. Train the **deployment model** (all 269 cleaned basins, seed 42) and simulate the dams  — ~3.6 h GPU
6. Monthly hm³ comparison vs `I_corr` → `bridge_results.csv` + figures

**Upload first to `MyDrive/pfe_rainfall/bridge/`:** `embalse.csv`, `bridge_inflows_monthly.csv`.
Dam ids are prefixed `dam_` (gauge 5001 and dam 5001 are different objects!).


In [ ]:
!pip -q install earthengine-api pyflwdir rasterio geopandas shapely pyproj neuralhydrology xarray netcdf4 >/dev/null
import ee, os, time, glob, pickle, yaml, shutil, numpy as np, pandas as pd
import rasterio, geopandas as gpd, pyflwdir, xarray as xr
from rasterio.transform import rowcol
from rasterio import features
from shapely.geometry import shape
from shapely.ops import unary_union
from pathlib import Path
import torch; print("GPU:", torch.cuda.is_available())

In [ ]:
ee.Authenticate(); ee.Initialize(project="pfe-rainfall"); print("EE ready")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE="/content/drive/MyDrive/pfe_rainfall"
EXP=os.path.join(BASE,"expansion")
BR=os.path.join(BASE,"bridge"); os.makedirs(BR,exist_ok=True)
NH_POOLED=os.path.join(BASE,"nh_pooled")          # prebuilt dataset (same as k-fold runs)
ATTR_POOLED=os.path.join(BASE,"attributes_pooled.csv")

# The 8 NATURAL dams (bridge_dam_screening.csv verdicts)
DAMS={5001:"TRANCO DE BEAS",5012:"BEMBEZAR",5018:"GUADALMENA",5039:"QUIEBRAJANO",
      5050:"COLOMERA",5060:"SAN CLEMENTE",5062:"GUADALMELLATO",5139:"ARENOSO"}

def read_csv_smart(p,**k):
    for sep in [";",",","\t"]:
        for enc in ["latin-1","utf-8"]:
            try:
                d=pd.read_csv(p,sep=sep,encoding=enc,engine="python",**k)
                if d.shape[1]>1: return d
            except Exception: pass
    raise RuntimeError("parse "+p)
def to_float(s):
    if pd.isna(s): return np.nan
    try: return float(str(s).replace(",",".").strip())
    except: return np.nan
def dms_to_deg(v):
    v=to_float(v)
    if pd.isna(v): return np.nan
    sign=-1 if v<0 else 1; v=abs(v)
    ss=v%100; mm=(v//100)%100; dd=v//10000
    return sign*(dd+mm/60.0+ss/3600.0)

emb=read_csv_smart(os.path.join(BR,"embalse.csv")); emb.columns=[c.strip().lower() for c in emb.columns]
emb["ref_ceh"]=pd.to_numeric(emb["ref_ceh"],errors="coerce")
emb=emb[emb["ref_ceh"].isin(DAMS)].copy()
emb["lon"]=emb["longwgs84"].map(dms_to_deg); emb["lat"]=emb["latwgs84"].map(dms_to_deg)
D=emb[["ref_ceh","nom_embalse","lon","lat"]].dropna().reset_index(drop=True)
assert len(D)==8, f"expected 8 dams, got {len(D)}"
print(D.to_string(index=False))

## 1. Delineation — snap to MAX upstream area at the dam wall
Dams sit on the main stem: within ~1 km of the wall the max-`upa` cell IS the dam
outlet. No `suprest` exists for dams, so QC = manual sanity check of the printed
areas (compare to CHG/SNCZI technical sheets — fill `expected_km2` if you have them).

In [ ]:
dir_tif=None
for p in [os.path.join(EXP,"hysheds_dir_iberia.tif"),"/content/drive/MyDrive/pfe_hysheds_iberia/hysheds_dir_iberia.tif"]:
    if os.path.exists(p): dir_tif=p; break
assert dir_tif, "hysheds_dir_iberia.tif not found"
with rasterio.open(dir_tif) as ds:
    dirarr=ds.read(1); transform=ds.transform; nrow,ncol=dirarr.shape
valid={0,1,2,4,8,16,32,64,128}
a=dirarr.astype(np.int32); a[a==255]=0
a=np.where(np.isin(a,list(valid)),a,247).astype(np.uint8)
flw=pyflwdir.from_array(a,ftype="d8",transform=transform,latlon=True,cache=True)
upa=flw.upstream_area(unit="km2"); upa=np.where(upa<0,0,upa).reshape(nrow,ncol)
print("grid",a.shape)

# Cap de plausibilité par barrage : le snap ignore toute cellule dont l'aire amont
# dépasse le cap. Indispensable quand le barrage est proche du fleuve principal
# (cas ARENOSO : sans cap, le snap attrape le Guadalquivir à ~22 000 km²).
CAP_KM2={5139:2000}          # ajouter d'autres caps si un barrage échoue au QC ci-dessous
DEFAULT_CAP=6000

GPKG_DAMS=os.path.join(BR,"catchments_dams.gpkg")
# >>> SUPPRIMER l'ancien gpkg sur Drive avant de relancer (il contient l'Arenoso faux) <<<
if os.path.exists(GPKG_DAMS):
    catd=gpd.read_file(GPKG_DAMS); print("cached:",len(catd),"polygons — supprimer bridge/catchments_dams.gpkg pour re-délinéer")
else:
    WIN=12
    polys=[]; qc=[]
    for r in D.itertuples():
        cap=CAP_KM2.get(int(r.ref_ceh),DEFAULT_CAP)
        r0,c0=rowcol(transform,r.lon,r.lat); cands=[]
        for dr in range(-WIN,WIN+1):
            for dc in range(-WIN,WIN+1):
                rr,cc=r0+dr,c0+dc
                if 0<=rr<nrow and 0<=cc<ncol and upa[rr,cc]>0:
                    cands.append((upa[rr,cc],rr,cc))
        cands.sort(reverse=True)
        print(f"{DAMS[r.ref_ceh]}: top candidats upa (km²) = {[round(c[0],1) for c in cands[:6]]} | cap={cap}")
        picked=next(((aa,rr,cc) for aa,rr,cc in cands if aa<=cap),None)
        assert picked, f"{DAMS[r.ref_ceh]}: aucun candidat sous le cap {cap} — élargir WIN ou revoir cap"
        aa,rr,cc=picked
        bas=flw.basins(idxs=np.array([rr*ncol+cc],dtype=np.int64)); m=bas>0
        sh=[shape(s) for s,v in features.shapes(m.astype("uint8"),mask=m,transform=transform) if v==1]
        polys.append(dict(indroea=f"dam_{r.ref_ceh}",dam=DAMS[r.ref_ceh],geometry=unary_union(sh)))
        qc.append(dict(ref_ceh=r.ref_ceh,dam=DAMS[r.ref_ceh],upa_snap_km2=round(aa,1)))
    catd=gpd.GeoDataFrame(polys,geometry="geometry",crs="EPSG:4326")
    catd["area_km2"]=catd.to_crs("EPSG:3035").area.values/1e6
    catd.to_file(GPKG_DAMS,driver="GPKG")
    qcd=pd.DataFrame(qc).merge(catd[["dam","area_km2"]],on="dam")

    # QC AUTOMATIQUE : lame d'écoulement impliquée par l'apport reconstitué.
    # depth (mm/an) = I_corr_annuel (hm³) × 1000 / aire (km²). Plausible : 20–600 mm/an
    # en contexte semi-aride. Une aire fausse se voit immédiatement ici.
    try:
        mm=pd.read_csv(os.path.join(BR,"bridge_inflows_monthly.csv"),parse_dates=["date"])
        ann=(mm[mm["n_valid"]>=25].groupby("ref_ceh")["I_corr"].mean()*12).rename("I_ann_hm3")
        qcd=qcd.merge(ann,on="ref_ceh",how="left")
        qcd["runoff_depth_mm"]=(qcd["I_ann_hm3"]*1000/qcd["area_km2"]).round(1)
        qcd["QC"]=np.where((qcd["runoff_depth_mm"]<20)|(qcd["runoff_depth_mm"]>600),"SUSPECT","ok")
    except Exception as e: print("QC inflow skipped:",e)
    print(qcd.to_string(index=False))
    print("\nTout 'SUSPECT' = aire probablement fausse -> ajuster CAP_KM2 et re-délinéer.")

## 2. Forcings — ERA5-Land yearly batched (same recipe as pooled_forcings_batched)

In [ ]:
RAW_FOLDER="pfe_bridge_forc"; RAW_DIR="/content/drive/MyDrive/"+RAW_FOLDER
FORC_D=os.path.join(BR,"forcings_dams"); os.makedirs(FORC_D,exist_ok=True)
YEAR_START,YEAR_END,SCALE=1960,2021,11132
catd=gpd.read_file(GPKG_DAMS)
cat_s=catd.copy(); cat_s["geometry"]=cat_s.geometry.simplify(0.004)
fc=ee.FeatureCollection([ee.Feature(ee.Geometry(r.geometry.__geo_interface__),{"indroea":r["indroea"]})
                         for _,r in cat_s.iterrows()])
era5=ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR").select(
    ["total_precipitation_sum","temperature_2m_min","temperature_2m_max"])
def export_year(Y):
    coll=era5.filterDate(f"{Y}-01-01",f"{Y+1}-01-01")
    def per_img(img):
        d=img.date().format("YYYY-MM-dd")
        return img.reduceRegions(collection=fc,reducer=ee.Reducer.mean(),
                                 scale=SCALE,tileScale=4).map(lambda f: f.set("date",d))
    out=coll.map(per_img).flatten()
    t=ee.batch.Export.table.toDrive(collection=out,description=f"brforc_{Y}",
        folder=RAW_FOLDER,fileNamePrefix=f"brforc_{Y}",fileFormat="CSV",
        selectors=["indroea","date","total_precipitation_sum","temperature_2m_min","temperature_2m_max"])
    t.start(); return t
tasks={}
for Y in range(YEAR_START,YEAR_END+1):
    if os.path.exists(os.path.join(RAW_DIR,f"brforc_{Y}.csv")): continue
    tasks[Y]=export_year(Y)
print("started",len(tasks),"tasks; polling...")
while tasks:
    done=[y for y,t in tasks.items() if t.status()["state"] in ("COMPLETED","FAILED","CANCELLED")]
    for y in done: print(" ",y,tasks[y].status()["state"]); tasks.pop(y)
    if tasks: time.sleep(30)
print("exports done")

In [ ]:
drive.mount('/content/drive',force_remount=True)
files=sorted(glob.glob(os.path.join(RAW_DIR,"brforc_*.csv"))); print("year files:",len(files))
df=pd.concat([pd.read_csv(f) for f in files],ignore_index=True)
df=df.rename(columns={"total_precipitation_sum":"pr_m","temperature_2m_min":"tmin_k","temperature_2m_max":"tmax_k"})
df["date"]=pd.to_datetime(df["date"])
cent=catd.to_crs(3035).geometry.centroid.to_crs(4326)
lat_map=dict(zip(catd["indroea"],cent.y))
def hargreaves(dates,tmin,tmax,tmean,lat):
    J=pd.to_datetime(dates).dt.dayofyear.values; phi=np.radians(lat)
    dr=1+0.033*np.cos(2*np.pi*J/365.0); dec=0.409*np.sin(2*np.pi*J/365.0-1.39)
    ws=np.arccos(np.clip(-np.tan(phi)*np.tan(dec),-1,1))
    Ra=(24*60/np.pi)*0.0820*dr*(ws*np.sin(phi)*np.sin(dec)+np.cos(phi)*np.cos(dec)*np.sin(ws))
    pet=0.0023*(Ra*0.408)*(tmean+17.8)*np.sqrt(np.clip(tmax-tmin,0,None))
    return np.clip(pet,0,None)
for bid,g in df.groupby("indroea"):
    g=g.sort_values("date")
    o=pd.DataFrame({"date":g["date"]})
    o["prcp_mm"]=pd.to_numeric(g["pr_m"],errors="coerce").values*1000.0
    o["tmin_c"]=pd.to_numeric(g["tmin_k"],errors="coerce").values-273.15
    o["tmax_c"]=pd.to_numeric(g["tmax_k"],errors="coerce").values-273.15
    o["tmean_c"]=(o["tmin_c"]+o["tmax_c"])/2.0
    o["pet_mm"]=hargreaves(o["date"],o["tmin_c"].values,o["tmax_c"].values,o["tmean_c"].values,lat_map[bid])
    o.to_csv(os.path.join(FORC_D,f"forcing_{bid}.csv"),index=False)
print("forcing CSVs:",len(glob.glob(os.path.join(FORC_D,'forcing_*.csv'))))

## 3. Static attributes (same recipe as pooled_attributes — only the 8 model STATICS matter)

In [ ]:
rows=[]
for fp in sorted(glob.glob(os.path.join(FORC_D,"forcing_*.csv"))):
    bid=os.path.basename(fp).replace("forcing_","").replace(".csv","")
    f=pd.read_csv(fp); f["date"]=pd.to_datetime(f["date"])
    p=f["prcp_mm"].values; pet=f["pet_mm"].values; tm=f["tmean_c"].values
    pm=np.nanmean(p); mon=f.groupby(f["date"].dt.month)["prcp_mm"].mean()
    rows.append(dict(indroea=bid,p_mean=round(pm,3),pet_mean=round(np.nanmean(pet),3),
        aridity=round(np.nanmean(pet)/pm,3) if pm>0 else np.nan,
        frac_snow=round(np.nansum(p[tm<0])/np.nansum(p),4) if np.nansum(p)>0 else 0,
        p_seasonality=round((mon.max()-mon.min())/mon.mean(),3) if mon.mean()>0 else np.nan))
clim=pd.DataFrame(rows)
cat_s=catd.copy(); cat_s["geometry"]=cat_s.geometry.simplify(0.003)
fc=ee.FeatureCollection([ee.Feature(ee.Geometry(r.geometry.__geo_interface__),{"indroea":r["indroea"]})
                         for _,r in cat_s.iterrows()])
elev=ee.Image("MERIT/Hydro/v1_0_1").select("elv").rename("elev_mean")
slope=ee.Terrain.slope(elev).rename("slope_mean")
wc=ee.ImageCollection("ESA/WorldCover/v200").first()
forest=wc.eq(10).rename("forest_frac")
img=elev.addBands([slope,forest])
res=img.reduceRegions(collection=fc,reducer=ee.Reducer.mean(),scale=250).map(lambda f:f.setGeometry(None))
gee=pd.DataFrame([d["properties"] for d in res.getInfo()["features"]])
attd=clim.merge(gee,on="indroea").merge(catd[["indroea","area_km2"]],on="indroea")
attd=attd.round(3); attd.to_csv(os.path.join(BR,"attributes_dams.csv"),index=False)
print(attd.to_string(index=False))
print("\nCheck: are these inside the training envelope (area 20-3500, aridity range of the pool)?")

## 4. Extend nh_pooled with the dam basins (qobs = NaN — simulation only)

In [ ]:
NH_BR=os.path.join(BR,"nh_bridge")
if os.path.exists(NH_BR): shutil.rmtree(NH_BR)
shutil.copytree(NH_POOLED,NH_BR)
tsdir=os.path.join(NH_BR,"time_series")
for fp in sorted(glob.glob(os.path.join(FORC_D,"forcing_*.csv"))):
    bid=os.path.basename(fp).replace("forcing_","").replace(".csv","")
    f=pd.read_csv(fp); f["date"]=pd.to_datetime(f["date"])
    ds=xr.Dataset({"prcp_mm":("date",f["prcp_mm"].values),"tmin_c":("date",f["tmin_c"].values),
                   "tmax_c":("date",f["tmax_c"].values),"pet_mm":("date",f["pet_mm"].values),
                   "qobs_mm":("date",np.full(len(f),np.nan))},coords={"date":f["date"].values})
    ds.to_netcdf(os.path.join(tsdir,bid+".nc"))
# attributes: NH lit les attributs dans le SOUS-DOSSIER attributes/ — chercher récursivement
att_files=glob.glob(os.path.join(NH_BR,"attributes","*.csv"))+ \
          [p for p in glob.glob(os.path.join(NH_BR,"*.csv")) if "attribute" in os.path.basename(p).lower()]
assert att_files, "aucun fichier d'attributs trouvé dans nh_bridge — structure inattendue" 
attd=pd.read_csv(os.path.join(BR,"attributes_dams.csv"))
for af in att_files:
    base=pd.read_csv(af); idc=base.columns[0]; base[idc]=base[idc].astype(str)
    add=attd.rename(columns={"indroea":idc}); add[idc]=add[idc].astype(str)
    add=add[~add[idc].isin(base[idc])].reindex(columns=base.columns)
    if len(add): pd.concat([base,add],ignore_index=True).to_csv(af,index=False)
    chk=pd.read_csv(af); chk[idc]=chk[idc].astype(str)
    missing=[b for b in attd["indroea"].astype(str) if b not in set(chk[idc])]
    assert not missing, f"{af}: barrages absents des attributs: {missing}"
    print("extended+verified",af)
print("nh_bridge ready:",len(glob.glob(os.path.join(tsdir,'*.nc'))),"basins")

## 5. Modèle de déploiement — 3 comptes, 1 seed chacun (~3,6 h/run)
Entraîné sur les 269 bassins nettoyés ; testé sur les **8 barrages ET les 269 jauges**
(l'éval jauges en 2009–2021 fournit le TEMPOREL-269 — résout TABLE_COHERENCE ligne 4).

| Compte | SEED |
|---|---|
| A | `42` |
| B | `1042` |
| C | `2042` |

Chaque compte édite SEED, Run All, puis télécharge `deploy_s<seed>.parquet` +
`temporal269_s<seed>.csv`. Le §6 (comparaison) se lance quand les 3 parquets sont
dans `bridge/sims/` — il moyenne les simulations des 3 seeds avant toute métrique
(même convention que le chiffre PUB de la thèse).

In [ ]:
SEED=42        # <<< A:42  B:1042  C:2042 — SEULE ligne à éditer
WORK="/content/work"; RUNS=os.path.join(WORK,"runs"); CFGD=os.path.join(WORK,"cfg")
SIMS_BR=os.path.join(BR,"sims")
for d in [RUNS,CFGD,SIMS_BR]: os.makedirs(d,exist_ok=True)
DYNAMIC=["prcp_mm","tmin_c","tmax_c","pet_mm"]; TARGET=["qobs_mm"]
STATICS=["area_km2","elev_mean","slope_mean","forest_frac","aridity","p_mean","frac_snow","p_seasonality"]
TRAIN=("01/10/1961","30/09/2008"); VAL=("01/10/1990","30/09/2008"); TEST=("01/10/1961","30/09/2021")
EPOCHS=20; HIDDEN=128; SEQ=365; BATCH=256; DROPOUT=0.5
BAD=["7124","8103","8148"]+['2034','2068','2070','2078','2080','2103','3002','3218','3225','3226',
     '3229','3233','3236','4156','4224','5050','7040','9066','9113']
att=pd.read_csv(ATTR_POOLED); att["indroea"]=att["indroea"].astype(str)
ALL=sorted([b for b in att["indroea"] if b not in BAD
            and os.path.exists(os.path.join(NH_BR,"time_series",b+".nc"))])
DAM_IDS=sorted([f"dam_{r}" for r in DAMS])
TEST_IDS=DAM_IDS+ALL           # barrages + jauges (temporel-269 gratuit)
print("train:",len(ALL),"| test:",len(TEST_IDS),"(dont",len(DAM_IDS),"barrages)")

out_parquet=os.path.join(SIMS_BR,f"deploy_s{SEED}.parquet")
if os.path.exists(out_parquet):
    print("cached — skip")
else:
    trf=os.path.join(CFGD,"tr.txt"); tef=os.path.join(CFGD,"te.txt")
    open(trf,"w").write("\n".join(ALL)); open(tef,"w").write("\n".join(TEST_IDS))
    cfg=dict(experiment_name=f"deploy_s{SEED}",run_dir=RUNS,train_basin_file=trf,validation_basin_file=trf,
        test_basin_file=tef,train_start_date=TRAIN[0],train_end_date=TRAIN[1],validation_start_date=VAL[0],
        validation_end_date=VAL[1],test_start_date=TEST[0],test_end_date=TEST[1],dataset="generic",data_dir=NH_BR,
        dynamic_inputs=DYNAMIC,target_variables=TARGET,static_attributes=STATICS,seq_length=SEQ,model="cudalstm",
        hidden_size=HIDDEN,initial_forget_bias=3,output_dropout=DROPOUT,output_activation="linear",
        head="regression",loss="NSE",optimizer="Adam",
        learning_rate={0:1e-3,int(EPOCHS*0.5):5e-4,int(EPOCHS*0.8):1e-4},batch_size=BATCH,epochs=EPOCHS,
        clip_gradient_norm=1,predict_last_n=1,num_workers=2,device="cuda:0",metrics=["NSE","KGE"],
        validate_every=EPOCHS,validate_n_random_basins=0,save_validation_results=False,seed=SEED)
    cfgp=os.path.join(CFGD,"deploy.yml"); yaml.safe_dump(cfg,open(cfgp,"w"),sort_keys=False)
    from neuralhydrology.nh_run import start_run, eval_run
    start_run(config_file=Path(cfgp))
    rd=sorted(Path(RUNS).glob(f"deploy_s{SEED}_*"))[-1]; eval_run(run_dir=rd,period="test")
    p=sorted(rd.glob("test/model_epoch*/test_results.p"))[-1]; res=pickle.load(open(p,"rb"))
    fr=[]; temp=[]
    def _kge(o,s):
        m=~(np.isnan(o)|np.isnan(s)); o,s=o[m],s[m]
        if len(o)<365 or o.std()==0 or s.std()==0: return np.nan
        r=np.corrcoef(o,s)[0,1]
        return 1-np.sqrt((r-1)**2+(s.std()/o.std()-1)**2+(s.mean()/o.mean()-1)**2)
    for bas,v in res.items():
        x=v["1D"]["xr"]
        df=pd.DataFrame({"basin":bas,"date":pd.to_datetime(x["date"].values),
            "obs":x["qobs_mm_obs"].values.squeeze(),"sim":x["qobs_mm_sim"].values.squeeze()})
        if str(bas).startswith("dam_"):
            fr.append(df)                              # barrages : série complète
        else:                                          # jauges : métrique temporelle 2009-2021
            g=df[df["date"]>="2009-10-01"]
            nse=np.nan
            o,s=g["obs"].values,g["sim"].values; m=~(np.isnan(o)|np.isnan(s))
            if m.sum()>365: nse=1-np.nansum((o[m]-s[m])**2)/np.nansum((o[m]-o[m].mean())**2)
            temp.append(dict(basin=bas,KGE=_kge(g["obs"].values,g["sim"].values),NSE=nse))
    pd.concat(fr,ignore_index=True).to_parquet(out_parquet)
    pd.DataFrame(temp).to_csv(os.path.join(SIMS_BR,f"temporal269_s{SEED}.csv"),index=False)
    t=pd.DataFrame(temp)
    print(f"TEMPOREL-269 (seed {SEED}, 2009-2021): median KGE {t['KGE'].median():.3f} / NSE {t['NSE'].median():.3f}")
    shutil.make_archive(os.path.join(BR,f"deploy_model_s{SEED}"),"zip",rd)
    print("saved dam sims + temporal metrics + model zip")

## 6. La table du jury — hm³ mensuels, ensemble 3 seeds vs apport reconstitué
Moyenne des simulations des 3 seeds AVANT métrique (même convention que la thèse).
Mois avec < 25 jours valides exclus. À lancer quand les 3 `deploy_s*.parquet` sont dans `bridge/sims/`.

In [ ]:
import matplotlib.pyplot as plt
parqs=sorted(glob.glob(os.path.join(SIMS_BR,"deploy_s*.parquet")))
assert len(parqs)==3, f"il faut les 3 seeds, trouvé {len(parqs)} — pas de table partielle (règle 3)"
sims=pd.concat([pd.read_parquet(p) for p in parqs],ignore_index=True)
sims=sims.groupby(["basin","date"],as_index=False)[["sim"]].mean()   # moyenne des 3 seeds
sims["ref_ceh"]=sims["basin"].str.replace("dam_","").astype(int)
area=dict(zip(catd["indroea"],catd["area_km2"]))
sims["sim_hm3"]=sims.apply(lambda r: r["sim"]*area[r["basin"]]/1000.0,axis=1)
sm=(sims.set_index("date").groupby("ref_ceh")["sim_hm3"]
        .resample("MS").agg(["sum","count"]).reset_index()
        .rename(columns={"sum":"sim_hm3","count":"n_sim"}))
obs=pd.read_csv(os.path.join(BR,"bridge_inflows_monthly.csv")); obs["date"]=pd.to_datetime(obs["date"])
m=obs.merge(sm,on=["ref_ceh","date"],how="inner")
m=m[(m["n_valid"]>=25)&(m["n_sim"]>=25)&m["I_corr"].notna()]

def kge(o,s):
    o,s=np.asarray(o,float),np.asarray(s,float)
    msk=~(np.isnan(o)|np.isnan(s)); o,s=o[msk],s[msk]
    if len(o)<24 or o.std()==0 or s.std()==0: return np.nan
    r=np.corrcoef(o,s)[0,1]
    return 1-np.sqrt((r-1)**2+(s.std()/o.std()-1)**2+(s.mean()/o.mean()-1)**2)
res=[]
for ref,g in m.groupby("ref_ceh"):
    o,s=g["I_corr"].values,g["sim_hm3"].values
    res.append(dict(ref_ceh=ref,dam=DAMS[ref],n_months=len(g),
        KGE_monthly=round(kge(o,s),3),
        NSE_monthly=round(1-np.nansum((o-s)**2)/np.nansum((o-np.nanmean(o))**2),3),
        pbias_pct=round(100*(np.nansum(s)-np.nansum(o))/np.nansum(o),1),
        mean_annual_obs_hm3=round(np.nanmean(o)*12,1),
        mean_annual_sim_hm3=round(np.nanmean(s)*12,1)))
R=pd.DataFrame(res).sort_values("KGE_monthly",ascending=False)
R.to_csv(os.path.join(BR,"bridge_results.csv"),index=False)
print(R.to_string(index=False))

fig,axes=plt.subplots(4,2,figsize=(14,12),sharex=False)
for ax,(ref,g) in zip(axes.ravel(),m.groupby("ref_ceh")):
    g=g.sort_values("date")
    ax.plot(g["date"],g["I_corr"],lw=.8,label="I_corr (reconstruit)")
    ax.plot(g["date"],g["sim_hm3"],lw=.8,label="LSTM (non jaugé)")
    ax.set_title(f"{DAMS[ref]} ({ref})",fontsize=9); ax.set_ylabel("hm³/mois")
axes[0,0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(BR,"bridge_monthly_series.png"),dpi=150)
print("wrote bridge_results.csv + bridge_monthly_series.png")